# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Andrew417/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Signal 1: staleness (`freshness_tier`) → declining share**

Belief: the staler a page, the more likely it is declining.

Verdict: **MIXED**

Reason: the declining share rises with staleness across the two big tiers, from 0.511 (`0-30`, n = 20,480) to 0.611 (`91-180`, n = 9,171), against a base rate of 0.542. `31-90` also sits above the base rate (0.589), but its n is only 175. The pattern breaks at the stalest tier: `181+` falls to 0.471 (n = 174), below the base rate. Both small tiers are weak evidence, so the verdict rests mostly on the two big tiers.

Takeaway: staleness up to about 180 days is a usable but partial signal. It cannot carry the rule alone, so it needs a partner signal, such as visibility (impressions).

**Rule (plain words):** refresh a page if it is stale (not updated for 91+ days) and visible (300+ impressions in 90 days). Rank by impressions, because more impressions means more clicks to win back.

Score = stale × visible × impressions_90d. Reason code: `stale_but_visible`. Action: `refresh_content`.

Cutoffs are my own choices, not repo requirements: 91 days starts the `91-180` tier (highest declining share among large tiers); 300 impressions starts the data dictionary's `moderate` tier.

In [32]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining"] = df["trend_direction"] == "down"
print(df.shape)
print(df["is_declining"].mean())
pd.set_option("display.max_columns", None)
df.head()

(30000, 45)
0.5420666666666667


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,provider_used,model_used,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,engaged_sessions_90d,ai_sessions_90d,scroll_events_90d,days_with_impressions,days_with_sessions,impressions_last_30d,clicks_last_30d,sessions_last_30d,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,content_age_days,age_tier,age_tier_order,days_since_last_update,freshness_tier,word_count_tier,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,NaN,gemini-2.5-flash,3803,29,22,17,16,1,0,1,88,13,578,2,2,987,13,9,187,181-365,5,20,0-30,2000-3500,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,True
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,NaN,gemini-3-flash-preview,15320,7,10,9,9,0,0,1,88,9,2501,2,3,5915,1,2,445,365+,6,25,0-30,2000-3500,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,True
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,NaN,gemini-2.5-flash,12581,11,14,11,11,0,0,4,88,11,2382,1,1,6089,3,3,141,91-180,4,20,0-30,3500+,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,True
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,NaN,NaN,11751,58,87,78,75,1,0,3,88,51,3626,22,35,4206,17,26,463,365+,6,22,0-30,NaN,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,False
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,NaN,gemini-3-flash-preview,19140,24,177,145,144,0,0,43,88,33,4211,10,14,6452,2,9,263,181-365,5,14,0-30,2000-3500,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,True


In [33]:
freshness_table  = df.groupby("freshness_tier")["is_declining"].agg(["mean" , "count"])
freshness_table  = freshness_table.reindex(["0-30", "31-90" , "91-180", "181+"])
print(freshness_table)

                    mean  count
freshness_tier                 
0-30            0.511377  20480
31-90           0.588571    175
91-180          0.611057   9171
181+            0.471264    174


In [34]:
df_with_position = df[df["avg_position"] > 0]
position_table = df_with_position.groupby("position_tier")["ctr"].agg(["median", "count"])
position_table = position_table.reindex(["top_3", "page_1", "striking", "page_3_5", "deep"])
print(position_table)

volume_table = df_with_position.groupby("position_tier")["impressions_90d"].median()
volume_table = volume_table.reindex(["top_3", "page_1", "striking", "page_3_5", "deep"])
print(volume_table)


               median  count
position_tier               
top_3            0.00   1116
page_1           0.16  11814
striking         0.11   7304
page_3_5         0.03   7242
deep             0.00   1319
position_tier
top_3         53.0
page_1      1179.5
striking     874.5
page_3_5     811.5
deep         218.0
Name: impressions_90d, dtype: float64


In [35]:
df_measurable = df_with_position[df_with_position["impressions_90d"] >= 100]
print(len(df_measurable))

position_table = df_measurable.groupby("position_tier")["ctr"].agg(["median", "count"])
position_table = position_table.reindex(["top_3", "page_1", "striking", "page_3_5", "deep"])
print(position_table)

22006
               median  count
position_tier               
top_3            0.19    533
page_1           0.23   8633
striking         0.15   5903
page_3_5         0.06   6058
deep             0.00    879


**Signal 2: CTR vs. position (`position_tier` → median `ctr`)**

Belief (from the lecture): the same CTR means different things at different positions, and better positions earn higher CTR. So CTR must be judged against pages at a similar position.

Verdict: **MIXED**

Method: I dropped the 1,205 rows with `avg_position = 0` (no position data, not rank zero) and kept only pages with at least 100 impressions in 90 days. That left 22,006 rows.

Reason: from `page_1` down, median CTR falls as position worsens: 0.23 (n = 8,633), 0.15 (`striking`, n = 5,903), 0.06 (`page_3_5`, n = 6,058), 0.00 (`deep`, n = 879). That part supports the belief. It breaks at the top: `top_3` is 0.19 (n = 533), below `page_1`. `top_3` is a thin tier in this slice (533 rows), and the data dictionary warns that its CTR is unreliable at low volume, so I don't treat the break as proof against the belief.

Takeaway: CTR does drop with worse position, so a CTR check needs a position comparison. The top tier is too thin to trust, so position peers should be `page_1` and below.

In [36]:
impression_table = df.groupby("impression_tier")["is_declining"].agg(["mean", "count"])
impression_table = impression_table.reindex(["none", "low", "moderate", "good", "excellent"])
print(impression_table)

                     mean    count
impression_tier                   
none                  NaN      NaN
low              0.453947  11248.0
moderate         0.614672  10469.0
good             0.586121   7205.0
excellent        0.461967   1078.0


**Signal 3: visibility (`impression_tier`) → declining share.** Belief: more visible pages are more worth acting on.

Verdict: **MIXED.** Share goes 0.454 (`low`, n = 11,248), 0.615 (`moderate`, n = 10,469), 0.586 (`good`, n = 7,205), 0.462 (`excellent`, n = 1,078), against a base rate of 0.542. It rises, then falls, and the gaps are 10+ points on large n. `none` is empty: every row has at least 1 impression.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [37]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

STALE_DAYS = 91
MIN_IMPRESSIONS = 300

df["is_stale"] = (df["days_since_last_update"] >= STALE_DAYS).astype(int)
df["is_visible"] = (df["impressions_90d"] >= MIN_IMPRESSIONS).astype(int)
df["score"] = df["is_stale"] * df["is_visible"] * df["impressions_90d"]

queue = df[df["score"] > 0].sort_values("score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1
queue["reason_code"] = "stale_but_visible"
queue["action"] = "refresh_content"

output_columns = ["rank", "content_id", "score", "reason_code", "action",
                  "days_since_last_update", "impressions_90d", "content_type"]
os.makedirs("../outputs", exist_ok=True)
queue[output_columns].to_csv("../outputs/baseline_action_score.csv", index=False)

print(len(queue), "pages in the queue")
queue[output_columns].head(10)

7234 pages in the queue


,rank,content_id,score,reason_code,action,days_since_last_update,impressions_90d,content_type
0,1,content_5fe46e04994d,517715,stale_but_visible,refresh_content,104,517715,keyword article
1,2,content_2dba2b1f9536,443434,stale_but_visible,refresh_content,104,443434,keyword article
2,3,content_2c2606c5d176,347399,stale_but_visible,refresh_content,104,347399,keyword article
3,4,content_cb112fce36be,309910,stale_but_visible,refresh_content,104,309910,keyword article
4,5,content_9532f197bbc8,309192,stale_but_visible,refresh_content,104,309192,keyword article
5,6,content_36ff89c8214e,295097,stale_but_visible,refresh_content,104,295097,keyword article
6,7,content_b28d1efd668f,286608,stale_but_visible,refresh_content,104,286608,keyword article
7,8,content_813e88069237,233561,stale_but_visible,refresh_content,104,233561,keyword article
8,9,content_c21024970297,211366,stale_but_visible,refresh_content,104,211366,keyword article
9,10,content_c8e9d6ab9013,208678,stale_but_visible,refresh_content,104,208678,keyword article


In [38]:
def precision_at_k(ranked_labels, k):
    return ranked_labels.iloc[:k].mean()

base_rate = df["is_declining"].mean()
for k in (20, 50, 100, 500):
    print(f"precision@{k}: {precision_at_k(queue['is_declining'], k):.3f}")
print(f"base rate: {base_rate:.3f}")

precision@20: 0.450
precision@50: 0.440
precision@100: 0.380
precision@500: 0.436
base rate: 0.542


**Result (observed):** precision@20 = 0.450, @50 = 0.440, @100 = 0.380, @500 = 0.436, base rate 0.542. The top of the queue is below random. But the whole queue's declining share is 0.618, above the base rate, so the stale-and-visible gate carries signal and the ranking by impressions is the weak part. I am not retuning: the baseline stays frozen so the Week-5 model has an honest bar to beat.

**Proxy note:** "declining" means impressions fell more than 20% (last 30 days vs. the 30 before). It stands in for "worth refreshing" and is not the same thing. Results are directional, decision-support only.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*
All ten have `days_since_last_update` = 104, are keyword articles, and have 208k+ impressions. Reason code for all: `stale_but_visible`. Action for all: `refresh_content`.

1. Rank 1 (517,715 impressions): declining. Wrong if the traffic is seasonal or it is a broad page where refreshing the text won't move rankings.
2. Rank 2 (443,434): NOT declining (weak pick). Wrong if the page is stable and only looks stale by date.
3. Rank 3 (347,399): declining. Wrong if the drop is a Google update or seasonality, not staleness.
4. Rank 4 (309,910): declining. Wrong if an update was made but not recorded in `days_since_last_update`.
5. Rank 5 (309,192): declining. Wrong if competitors gained on it for reasons a refresh can't fix.
6. Rank 6 (295,097): NOT declining (weak pick). Wrong if the page performs steadily, so refreshing risks breaking what works.
7. Rank 7 (286,608): NOT declining (weak pick). Wrong for the same reason: 104 days may not be old for this topic.
8. Rank 8 (233,561): declining. Wrong if the decline is seasonal.
9. Rank 9 (211,366): NOT declining (weak pick). Wrong if it is stable or growing and only ranks high because of raw volume.
10. Rank 10 (208,678): declining. Wrong if the loss is from a ranking change unrelated to freshness.

Confidence: low to moderate. Only 6 of the top 10 were declining (observed), and all ten share one staleness value, so the ranking within the top is driven only by impressions.

In [39]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
queue[["rank", "days_since_last_update", "impressions_90d", "content_type", "is_declining"]].head(10)

,rank,days_since_last_update,impressions_90d,content_type,is_declining
0,1,104,517715,keyword article,True
1,2,104,443434,keyword article,False
2,3,104,347399,keyword article,True
3,4,104,309910,keyword article,True
4,5,104,309192,keyword article,True
5,6,104,295097,keyword article,False
6,7,104,286608,keyword article,False
7,8,104,233561,keyword article,True
8,9,104,211366,keyword article,False
9,10,104,208678,keyword article,True


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks (observed):** 4 of the top 10 (ranks 2, 6, 7, 9) were not declining, and precision@K is below the base rate at every K I tried. All top-20 pages share one `days_since_last_update` value, so within the top the rule ranks purely by impressions.

 Top 20 (observed): 3 distinct clients, all 20 at 104 days since update, 9 of 20 declining. The queue overall is 0.618 declining. A single shared staleness value and only 3 clients suggest a batch update or one client dominating the top (my hypothesis, not tested).

**Leakage check:**
- Score inputs are only `days_since_last_update` and `impressions_90d`, confirmed by the assertion above.
- `trend_direction`, `trend_pct`, and the last/prev-30-day columns are not used. The label `is_declining` is used only to evaluate, never to score.
- No product flags or future-window columns are used.
- Caveat: `impressions_90d` covers the same 90 days that the label's two 30-day windows sit inside. The repo's own pipeline uses it as a feature, so I accept it here, but it is a possible partial overlap and I flag it honestly.

**What I would do next (not done here):** compare against peers at similar position, and check whether one client or one batch update explains the single staleness value in the top 20.


In [40]:
top_20 = queue.head(20)
print(top_20["client_id"].nunique(), "distinct clients in the top 20")
print(top_20["days_since_last_update"].value_counts())
print("top-20 declining:", int(top_20["is_declining"].sum()), "of 20")
print("queue declining share:", round(queue["is_declining"].mean(), 3))

label_or_future_columns = [
    "trend_direction", "trend_pct", "is_declining",
    "impressions_last_30d", "impressions_prev_30d",
    "clicks_last_30d", "clicks_prev_30d",
    "sessions_last_30d", "sessions_prev_30d",
]
score_inputs = ["days_since_last_update", "impressions_90d"]
assert not set(score_inputs) & set(label_or_future_columns), "leak: a forbidden column feeds the score"
print("score inputs:", score_inputs)

3 distinct clients in the top 20
days_since_last_update
104    20
Name: count, dtype: int64
top-20 declining: 9 of 20
queue declining share: 0.618
score inputs: ['days_since_last_update', 'impressions_90d']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.